# Hunde- und Katzenbilder laden und labeln

## Drive in Colab mounten

In [ ]:
use_colab = True

if (use_colab):
  # Google Drive mounten
  from google.colab import drive
  drive.mount("/content/drive")
  base_path = "/content/drive/MyDrive"
else:
  base_path = '..'

## Bilder einlesen

Quelle der Bilddateien:
https://downloads.codingcoursestv.eu/037%20-%20neuronale%20netze/PetImages.zip

### `image_dataset_from_directory()`

Mit `image_dataset_from_directory()` werden Bilder nicht vollständig in den Arbeitsspeicher geladen, sondern während des Trainings batchweise von der Festplatte eingelesen. Die Klassenlabels werden dabei automatisch aus den Ordnernamen (z. B. `Cat` und `Dog`) erzeugt. Dadurch können auch sehr große Bilddatensätze effizient mit `model.fit()` verarbeitet werden.

In [ ]:
import os
import tensorflow as tf


dataset_path = os.path.join(base_path, "Datensätze", "PetImages")

train_ds = tf.keras.utils.image_dataset_from_directory(
    dataset_path,
    validation_split=0.2,
    subset="training",
    seed=42,
    image_size=(224, 224),
    batch_size=32,
    label_mode="binary"
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    dataset_path,
    validation_split=0.2,
    subset="validation",
    seed=42,
    image_size=(224, 224),
    batch_size=32,
    label_mode="binary"
)

print(train_ds.class_names)

## Transfer Learning mit VGG16


### Die Bilder werden für das VGG16 vorverarbeitet
Mit **AUTOTUNE** entscheidet TensorFow automatisch wie viele Threads es für die Verarbeitung startet

In [ ]:
from tensorflow.keras.applications.vgg16 import VGG16, preprocess_input
AUTOTUNE = tf.data.AUTOTUNE

train_ds = train_ds.map(
    lambda x, y: (preprocess_input(tf.cast(x, tf.float32)), y),
    num_parallel_calls=AUTOTUNE
)

val_ds = val_ds.map(
    lambda x, y: (preprocess_input(tf.cast(x, tf.float32)), y),
    num_parallel_calls=AUTOTUNE
)

train_ds = train_ds.prefetch(AUTOTUNE)
val_ds = val_ds.prefetch(AUTOTUNE)

VGG16 ohne die Dense-Layer laden


In [ ]:
vgg16_model = VGG16(include_top=False, input_shape=(224, 224, 3))
vgg16_model.trainable = False # Die Parameter von VGG16 müssen nicht mehr trainiert werden.
vgg16_model.summary()

An die Vorverarbeitungsschichten von VGG16 hängen wir unsere Dense-Layer

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Flatten
from tensorflow.keras.optimizers import Adam

model = Sequential()
model.add(vgg16_model)

model.add(Flatten())
model.add(Dense(4096, activation="relu"))
model.add(Dense(1024, activation="relu"))
model.add(Dense(1, activation="sigmoid"))
model.compile(optimizer="rmsprop", loss="binary_crossentropy", metrics=["accuracy"])
#model.summary()

Model trainieren

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping

early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True
)

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10,
    callbacks=[early_stopping]
)

In [ ]:
# Auswertung auf den Validierungsdaten
loss, accuracy = model.evaluate(val_ds, verbose=1)

print(f"Validierungs-Loss: {loss:.4f}")
print(f"Validierungs-Accuracy: {accuracy:.4f}")